# 🍔 Food + Calorie Predictor (Lightweight Training)
This notebook trains a lightweight model and saves it for use in a Streamlit app.

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

In [4]:
img_size = (160, 160)
batch_size = 8

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "food-101/images",
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "food-101/images",
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)


class_names = train_ds.class_names
num_classes = len(class_names)
# 🔥 LIMIT DATA (prevents crash)
train_ds = train_ds.take(3000)
val_ds = val_ds.take(800)

print(class_names)

Found 101000 files belonging to 101 classes.
Using 80800 files for training.
Found 101000 files belonging to 101 classes.
Using 20200 files for validation.
['apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare', 'beet_salad', 'beignets', 'bibimbap', 'bread_pudding', 'breakfast_burrito', 'bruschetta', 'caesar_salad', 'cannoli', 'caprese_salad', 'carrot_cake', 'ceviche', 'cheese_plate', 'cheesecake', 'chicken_curry', 'chicken_quesadilla', 'chicken_wings', 'chocolate_cake', 'chocolate_mousse', 'churros', 'clam_chowder', 'club_sandwich', 'crab_cakes', 'creme_brulee', 'croque_madame', 'cup_cakes', 'deviled_eggs', 'donuts', 'dumplings', 'edamame', 'eggs_benedict', 'escargots', 'falafel', 'filet_mignon', 'fish_and_chips', 'foie_gras', 'french_fries', 'french_onion_soup', 'french_toast', 'fried_calamari', 'fried_rice', 'frozen_yogurt', 'garlic_bread', 'gnocchi', 'greek_salad', 'grilled_cheese_sandwich', 'grilled_salmon', 'guacamole', 'gyoza', 'hamburger', 'hot_and_sour_sou

In [5]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.shuffle(500).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [6]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [7]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

model = keras.Sequential([
    data_augmentation,
    layers.Rescaling(1./255),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes)
])

In [8]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [9]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Epoch 1/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 356s 116ms/step - accuracy: 0.2775 - loss: 3.0313 - val_accuracy: 0.3789 - val_loss: 2.5060
Epoch 2/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 340s 113ms/step - accuracy: 0.3941 - loss: 2.4304 - val_accuracy: 0.4061 - val_loss: 2.3981
Epoch 3/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 328s 109ms/step - accuracy: 0.4295 - loss: 2.2758 - val_accuracy: 0.4197 - val_loss: 2.3846
Epoch 4/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 331s 110ms/step - accuracy: 0.4474 - loss: 2.1720 - val_accuracy: 0.4325 - val_loss: 2.3341
Epoch 5/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 339s 112ms/step - accuracy: 0.4650 - loss: 2.1001 - val_accuracy: 0.4159 - val_loss: 2.3700


In [10]:
model.save("food_model.keras")

## 🍽️ Calorie Mapping
Update this dictionary based on your classes

In [11]:
calorie_dict = {
    'apple_pie': 237, 'baby_back_ribs': 292, 'baklava': 334,
    'beef_carpaccio': 120, 'beef_tartare': 180, 'beet_salad': 150,
    'beignets': 290, 'bibimbap': 490, 'bread_pudding': 300,
    'breakfast_burrito': 450, 'bruschetta': 150, 'caesar_salad': 180,
    'cannoli': 250, 'caprese_salad': 220, 'carrot_cake': 350,
    'ceviche': 200, 'cheese_plate': 400, 'cheesecake': 321,
    'chicken_curry': 240, 'chicken_quesadilla': 300, 'chicken_wings': 430,
    'chocolate_cake': 371, 'chocolate_mousse': 280, 'churros': 300,
    'clam_chowder': 200, 'club_sandwich': 350, 'crab_cakes': 240,
    'creme_brulee': 300, 'croque_madame': 500, 'cup_cakes': 305,
    'deviled_eggs': 160, 'donuts': 260, 'dumplings': 220,
    'edamame': 120, 'eggs_benedict': 420, 'escargots': 180,
    'falafel': 333, 'filet_mignon': 250, 'fish_and_chips': 600,
    'foie_gras': 460, 'french_fries': 365, 'french_onion_soup': 200,
    'french_toast': 350, 'fried_calamari': 300, 'fried_rice': 330,
    'frozen_yogurt': 150, 'garlic_bread': 250, 'gnocchi': 250,
    'greek_salad': 200, 'grilled_cheese_sandwich': 400, 'grilled_salmon': 280,
    'guacamole': 160, 'gyoza': 250, 'hamburger': 354,
    'hot_and_sour_soup': 150, 'hot_dog': 290, 'huevos_rancheros': 300,
    'hummus': 166, 'ice_cream': 207, 'lasagna': 350,
    'lobster_bisque': 250, 'lobster_roll_sandwich': 400,
    'macaroni_and_cheese': 310, 'macarons': 100, 'miso_soup': 84,
    'mussels': 230, 'nachos': 346, 'omelette': 154,
    'onion_rings': 276, 'oysters': 50, 'pad_thai': 350,
    'paella': 400, 'pancakes': 227, 'panna_cotta': 300,
    'peking_duck': 337, 'pho': 350, 'pizza': 285,
    'pork_chop': 250, 'poutine': 500, 'prime_rib': 400,
    'pulled_pork_sandwich': 450, 'ramen': 436, 'ravioli': 250,
    'red_velvet_cake': 360, 'risotto': 350, 'samosa': 262,
    'sashimi': 200, 'scallops': 200, 'seaweed_salad': 90,
    'shrimp_and_grits': 400, 'spaghetti_bolognese': 350,
    'spaghetti_carbonara': 380, 'spring_rolls': 200,
    'steak': 271, 'strawberry_shortcake': 300, 'sushi': 200,
    'tacos': 300, 'takoyaki': 300, 'tiramisu': 300,
    'tuna_tartare': 180, 'waffles': 291
}

In [12]:
from tensorflow.keras.preprocessing import image

def predict(img_path):
    img = image.load_img(img_path, target_size=(160,160))
    img_array = image.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)

    preds = model.predict(img_array)
    score = tf.nn.softmax(preds[0])

    food = class_names[np.argmax(score)]
    confidence = np.max(score)*100
    calories = calorie_dict.get(food, "Unknown")

    print(f"Prediction: {food}")
    print(f"Confidence: {confidence:.2f}%")
    print(f"Calories: {calories}")

In [13]:
import json

with open("class_names.json", "w") as f:
    json.dump(class_names, f)

print("Saved class names ✅")

Saved class names ✅
